In [ ]:
# pip install requests beautifulsoup4
# pip install "numpy==1.26.4"
# pip install onnxruntime
# pip install deutschland

In [ ]:
"""
Financial Report Scraper
========================
Downloads official financial filings for:

  US companies  (Tesla, Disney, Netflix)
    → 10-K  (annual report)      via SEC EDGAR free API
    → DEF 14A (proxy statement)  via SEC EDGAR free API

  German company (Volkswagen)
    → Jahresabschluss / Konzernabschluss
      PRIMARY:  `deutschland` package  (handles Bundesanzeiger CAPTCHA)
      FALLBACK: VW investor-relations page (direct PDF links, no CAPTCHA)

---------------------------------------------------------------------------
INSTALL — run these lines in order to avoid dependency conflicts:

    pip install requests beautifulsoup4
    pip install numpy==1.26.4
    pip install onnxruntime
    pip install deutschland

If `deutschland` still fails (e.g. no onnxruntime wheel for your platform),
the scraper automatically falls back to scraping VW's IR page directly.
No action needed — the fallback is built in.

---------------------------------------------------------------------------
SEC EDGAR requires a User-Agent header that identifies you.
Edit SEC_USER_AGENT below with your name and email before running.

Usage:
    python financial_scraper.py          # all companies, all years
    
    from financial_scraper import Scraper
    Scraper("Tesla", 2023).scrape()
    Scraper("Volkswagen", 2023).scrape()
"""

import os
import re
import time
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import requests
from bs4 import BeautifulSoup

# ---------------------------------------------------------------------------
# Config — edit SEC_USER_AGENT before running
# ---------------------------------------------------------------------------

COMPANIES: dict[str, str] = {
    "Tesla":      "US",
    "Disney":     "US",
    "Netflix":    "US",
    "Volkswagen": "DE",
}

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]

OUTPUT_DIR    = Path("financial_reports")
REQUEST_DELAY = 3.0   # SEC rate limit is 10 req/s; stay well under

# SEC requires "Name email@example.com" — replace with your own
SEC_USER_AGENT = "Katharina Meder katharina.meder@student.uni-halle.de"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

BROWSER_UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)


# ---------------------------------------------------------------------------
# Data model
# ---------------------------------------------------------------------------

@dataclass
class Filing:
    company:    str
    form_type:  str       # "10-K" | "DEF 14A" | "Jahresabschluss" | …
    year:       int
    title:      str
    url:        str
    source:     str       # "SEC_EDGAR" | "Bundesanzeiger" | "VW_IR"
    local_path: Optional[Path] = None
    _content:   str = "" # used for text-only Bundesanzeiger reports


# ---------------------------------------------------------------------------
# SEC EDGAR adapter — US companies (Tesla, Disney, Netflix)
# ---------------------------------------------------------------------------

class SECEdgarAdapter:
    """
    Uses the SEC EDGAR free API — no auth, no rate-limit key.

    Flow:
      1. Fetch company_tickers.json  →  resolve company name → CIK
      2. Fetch submissions/{CIK}.json  →  list all filings
      3. Filter by form type (10-K / DEF 14A) and fiscal year
      4. Build the direct document URL from the accession number
      5. Download the primary HTML document

    Docs: https://www.sec.gov/search-filings/edgar-application-programming-interfaces
    Required header: User-Agent "Name email@example.com"
    """

    TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
    SUBMISSIONS = "https://data.sec.gov/submissions/CIK{cik}.json"
    FORM_TYPES  = ["10-K", "DEF 14A"]

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": SEC_USER_AGENT,
            "Accept":     "application/json, text/html, */*",
        })
        self._ticker_cache: dict[str, str] = {}  # company name (lower) → CIK (zero-padded)

    # ------------------------------------------------------------------

    def get_filings(self, company: str, year: int) -> list[Filing]:
        cik = self._resolve_cik(company)
        if not cik:
            log.warning("SEC: could not resolve CIK for '%s'", company)
            return []

        recent = self._fetch_recent(cik)
        if not recent:
            return []

        return self._extract(company, year, cik, recent)

    # ------------------------------------------------------------------

    def _resolve_cik(self, company: str) -> Optional[str]:
        if not self._ticker_cache:
            self._load_tickers()
        needle = company.lower()
        for name, cik in self._ticker_cache.items():
            if needle in name:
                return cik
        return None

    def _load_tickers(self) -> None:
        try:
            r = self.session.get(self.TICKERS_URL, timeout=15)
            r.raise_for_status()
            for entry in r.json().values():
                name = entry.get("title", "").lower()
                cik  = str(entry.get("cik_str", "")).zfill(10)
                self._ticker_cache[name] = cik
            log.info("SEC: loaded %d tickers", len(self._ticker_cache))
        except requests.RequestException as exc:
            log.warning("SEC: ticker load failed: %s", exc)

    def _fetch_recent(self, cik: str) -> dict:
        url = self.SUBMISSIONS.format(cik=cik)
        try:
            time.sleep(REQUEST_DELAY)
            r = self.session.get(url, timeout=20)
            r.raise_for_status()
            return r.json().get("filings", {}).get("recent", {})
        except requests.RequestException as exc:
            log.warning("SEC: submissions failed for CIK %s: %s", cik, exc)
            return {}

    def _extract(self, company: str, year: int, cik: str, recent: dict) -> list[Filing]:
        forms       = recent.get("form", [])
        accessions  = recent.get("accessionNumber", [])
        dates       = recent.get("filingDate", [])
        docs        = recent.get("primaryDocument", [])

        filings = []
        for form, acc, date, doc in zip(forms, accessions, dates, docs):
            if form not in self.FORM_TYPES:
                continue
            # 10-Ks for FY2023 are filed in early 2024, so match year and year+1
            filed_year = int(date[:4]) if date else 0
            if filed_year not in (year, year + 1):
                continue

            acc_nodash = acc.replace("-", "")
            doc_url = (
                f"https://www.sec.gov/Archives/edgar/data/"
                f"{int(cik)}/{acc_nodash}/{doc}"
            )
            filings.append(Filing(
                company=company,
                form_type=form,
                year=year,
                title=f"{company} {form} — filed {date}",
                url=doc_url,
                source="SEC_EDGAR",
            ))

        log.info("SEC: %d filing(s) for '%s' (%d)", len(filings), company, year)
        return filings


# ---------------------------------------------------------------------------
# Bundesanzeiger adapter — Volkswagen (PRIMARY)
# ---------------------------------------------------------------------------

class BundesanzeigerAdapter:
    """
    Uses the `deutschland` package which wraps bundesanzeiger.de and solves
    its image CAPTCHA automatically via a bundled ML model (onnxruntime).

    Install:
        pip install numpy==1.26.4
        pip install onnxruntime
        pip install deutschland

    If the import fails, VWInvestorRelationsAdapter is used as fallback.
    """

    LEGAL_NAMES = {
        "Volkswagen": "Volkswagen Aktiengesellschaft",
    }

    def get_filings(self, company: str, year: int) -> list[Filing]:
        try:
            from deutschland.bundesanzeiger import Bundesanzeiger
        except ImportError as exc:
            log.warning(
                "Bundesanzeiger: `deutschland` not importable (%s). "
                "Falling back to VW IR page.", exc
            )
            return []

        legal = self.LEGAL_NAMES.get(company, company)
        log.info("Bundesanzeiger: searching '%s' (%d)…", legal, year)

        try:
            ba = Bundesanzeiger()
            all_reports: dict = ba.get_reports(legal)
        except Exception as exc:
            log.warning("Bundesanzeiger: get_reports failed: %s", exc)
            return []

        filings = []
        for title, content in all_reports.items():
            if str(year) not in title:
                continue
            form_type = (
                "Konzernabschluss" if "Konzernabschluss" in title
                else "Jahresabschluss"
            )
            filings.append(Filing(
                company=company,
                form_type=form_type,
                year=year,
                title=title.strip(),
                url="https://www.bundesanzeiger.de",
                source="Bundesanzeiger",
                _content=str(content),
            ))

        log.info("Bundesanzeiger: %d report(s) for '%s' (%d)", len(filings), company, year)
        return filings


# ---------------------------------------------------------------------------
# VW Investor Relations fallback — Volkswagen (FALLBACK)
# ---------------------------------------------------------------------------

class VWInvestorRelationsAdapter:
    """
    Scrapes VW's public investor-relations page for direct PDF links.
    No login, no CAPTCHA. Used when the `deutschland` package is unavailable.

    Source: https://www.volkswagen-group.com/de/finanzberichte-18134
    Lists Jahresabschluss and Konzernabschluss PDFs for each year.
    """

    IR_URL = "https://www.volkswagen-group.com/de/finanzberichte-18134"

    # Keywords in link text that indicate a financial report PDF
    REPORT_KEYWORDS = [
        "jahresabschluss", "konzernabschluss",
        "annual report", "annual financial",
    ]

    def get_filings(self, company: str, year: int) -> list[Filing]:
        log.info("VW IR: scraping investor-relations page for %d…", year)
        headers = {
            "User-Agent": BROWSER_UA,
            "Accept":     "text/html,application/xhtml+xml,*/*",
            "Accept-Language": "de-DE,de;q=0.9",
        }
        try:
            time.sleep(REQUEST_DELAY)
            r = requests.get(self.IR_URL, headers=headers, timeout=20)
            r.raise_for_status()
        except requests.RequestException as exc:
            log.warning("VW IR: page fetch failed: %s", exc)
            return []

        soup = BeautifulSoup(r.text, "html.parser")
        filings = []
        seen = set()

        for a in soup.find_all("a", href=True):
            href  = a["href"]
            text  = a.get_text(separator=" ", strip=True).lower()
            # Must mention the year and be a financial report type
            if str(year) not in text and str(year) not in href:
                continue
            if not any(kw in text for kw in self.REPORT_KEYWORDS):
                continue
            if href in seen:
                continue
            seen.add(href)

            if not href.startswith("http"):
                href = "https://www.volkswagen-group.com" + href

            form_type = (
                "Konzernabschluss" if "konzern" in text
                else "Jahresabschluss"
            )
            title = f"VW {form_type} {year}"
            filings.append(Filing(
                company=company,
                form_type=form_type,
                year=year,
                title=title,
                url=href,
                source="VW_IR",
            ))

        log.info("VW IR: %d report(s) found for %d", len(filings), year)
        return filings


# ---------------------------------------------------------------------------
# Downloader
# ---------------------------------------------------------------------------

class Downloader:
    """Downloads SEC HTML docs, VW PDFs, or saves Bundesanzeiger text."""

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": SEC_USER_AGENT,
            "Accept":     "text/html,application/pdf,*/*",
        })

    def save(self, filing: Filing, dest_dir: Path) -> Optional[Path]:
        dest_dir.mkdir(parents=True, exist_ok=True)
        safe = re.sub(r"[^\w\-]+", "_", filing.title)[:80]

        # Bundesanzeiger: content is already in memory as text
        if filing.source == "Bundesanzeiger":
            if not filing._content:
                return None
            dest = dest_dir / f"{safe}.txt"
            if not dest.exists():
                dest.write_text(filing._content, encoding="utf-8")
                log.info("    saved: %s", dest.name)
            else:
                log.info("    skip (exists): %s", dest.name)
            return dest

        # SEC / VW IR: download from URL
        time.sleep(REQUEST_DELAY)
        # VW IR pages need a browser UA
        if filing.source == "VW_IR":
            self.session.headers["User-Agent"] = BROWSER_UA

        try:
            r = self.session.get(
                filing.url, timeout=30, stream=True, allow_redirects=True
            )
            r.raise_for_status()
        except requests.RequestException as exc:
            log.warning("    download failed '%s': %s", filing.url, exc)
            return None

        ct  = r.headers.get("content-type", "")
        ext = ".pdf" if ("pdf" in ct or filing.url.lower().endswith(".pdf")) else ".htm"
        dest = dest_dir / f"{safe}{ext}"

        if dest.exists():
            log.info("    skip (exists): %s", dest.name)
        else:
            with open(dest, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            log.info("    saved: %s", dest.name)

        return dest


# ---------------------------------------------------------------------------
# Main Scraper class
# ---------------------------------------------------------------------------

class Scraper:
    """
    Fetches and saves financial filings for `company` in `year`.

    Routing:
      Tesla / Disney / Netflix  →  SEC EDGAR  (10-K + DEF 14A)
      Volkswagen                →  Bundesanzeiger via `deutschland` package,
                                   with automatic fallback to VW IR page

    Parameters
    ----------
    company : str
        One of: "Tesla", "Disney", "Netflix", "Volkswagen"
    year : int
        Fiscal year, e.g. 2023
    output_dir : Path | str
        Root save folder. Files land in <output_dir>/<company>/<year>/.
    """

    _sec   = SECEdgarAdapter()
    _banz  = BundesanzeigerAdapter()
    _vwir  = VWInvestorRelationsAdapter()
    _dl    = Downloader()

    def __init__(
        self,
        company: str,
        year: int,
        output_dir: Path | str = OUTPUT_DIR,
    ):
        if company not in COMPANIES:
            raise ValueError(
                f"Unknown company '{company}'. "
                f"Valid options: {list(COMPANIES.keys())}"
            )
        self.company    = company
        self.year       = year
        self.region     = COMPANIES[company]
        self.output_dir = Path(output_dir) / company / str(year)

        self.filings:     list[Filing] = []
        self.saved_paths: list[Path]   = []

    def scrape(self) -> list[Filing]:
        log.info("=== %s (%s, %d) ===", self.company, self.region, self.year)

        if self.region == "US":
            self.filings = self._sec.get_filings(self.company, self.year)

        else:
            # Try deutschland package first; fall back to VW IR page
            self.filings = self._banz.get_filings(self.company, self.year)
            if not self.filings:
                log.info("Falling back to VW investor-relations page…")
                self.filings = self._vwir.get_filings(self.company, self.year)

        for filing in self.filings:
            path = self._dl.save(filing, self.output_dir)
            if path:
                filing.local_path = path
                self.saved_paths.append(path)

        n = len(self.saved_paths)
        print(
            f"\n✓  {n} filing(s) for '{self.company}' ({self.year}) "
            f"→ {self.output_dir}"
        )
        return self.filings

    def __repr__(self) -> str:
        return f"Scraper(company={self.company!r}, year={self.year})"


# ---------------------------------------------------------------------------
# Bulk runner
# ---------------------------------------------------------------------------

def run_all(
    companies: list[str] = list(COMPANIES.keys()),
    years: list[int] = YEARS,
) -> None:
    total = 0
    for company in companies:
        for year in years:
            s = Scraper(company, year)
            s.scrape()
            total += len(s.saved_paths)
    print(f"\n{'='*55}\nDone — {total} file(s) saved in total.")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    run_all()

10:37:35  INFO      === Tesla (US, 2020) ===
10:37:37  WARNING   SEC: ticker load failed: 404 Client Error: Not Found for url: https://data.sec.gov/files/company_tickers.json
10:37:37  WARNING   SEC: could not resolve CIK for 'Tesla'
10:37:37  INFO      === Tesla (US, 2021) ===
10:37:37  WARNING   SEC: ticker load failed: 404 Client Error: Not Found for url: https://data.sec.gov/files/company_tickers.json
10:37:37  WARNING   SEC: could not resolve CIK for 'Tesla'
10:37:37  INFO      === Tesla (US, 2022) ===



✓  0 filing(s) for 'Tesla' (2020) → financial_reports\Tesla\2020

✓  0 filing(s) for 'Tesla' (2021) → financial_reports\Tesla\2021


10:37:37  WARNING   SEC: ticker load failed: 404 Client Error: Not Found for url: https://data.sec.gov/files/company_tickers.json
10:37:37  WARNING   SEC: could not resolve CIK for 'Tesla'
10:37:37  INFO      === Tesla (US, 2023) ===
10:37:38  WARNING   SEC: ticker load failed: 404 Client Error: Not Found for url: https://data.sec.gov/files/company_tickers.json
10:37:38  WARNING   SEC: could not resolve CIK for 'Tesla'
10:37:38  INFO      === Tesla (US, 2024) ===



✓  0 filing(s) for 'Tesla' (2022) → financial_reports\Tesla\2022

✓  0 filing(s) for 'Tesla' (2023) → financial_reports\Tesla\2023


10:37:38  WARNING   SEC: ticker load failed: 404 Client Error: Not Found for url: https://data.sec.gov/files/company_tickers.json
10:37:38  WARNING   SEC: could not resolve CIK for 'Tesla'
10:37:38  INFO      === Tesla (US, 2025) ===
10:37:38  WARNING   SEC: ticker load failed: 404 Client Error: Not Found for url: https://data.sec.gov/files/company_tickers.json
10:37:38  WARNING   SEC: could not resolve CIK for 'Tesla'
10:37:38  INFO      === Disney (US, 2020) ===


...


✓  7 filing(s) for 'Volkswagen' (2023) → financial_reports\Volkswagen\2023


10:39:19  INFO      VW IR: 8 report(s) found for 2024
10:39:25  INFO          saved: VW_Konzernabschluss_2024.pdf
10:39:29  INFO          saved: VW_Jahresabschluss_2024.pdf
10:39:33  INFO          saved: VW_Konzernabschluss_2024.htm
10:39:37  INFO          saved: VW_Jahresabschluss_2024.htm
10:39:40  INFO          skip (exists): VW_Konzernabschluss_2024.pdf
10:39:44  INFO          skip (exists): VW_Jahresabschluss_2024.pdf
10:39:48  INFO          skip (exists): VW_Konzernabschluss_2024.htm
10:39:52  INFO          skip (exists): VW_Jahresabschluss_2024.htm
10:39:52  INFO      === Volkswagen (DE, 2025) ===
10:39:52  WARNING   Bundesanzeiger: `deutschland` not importable (No module named 'deutschland'). Falling back to VW IR page.
10:39:52  INFO      Falling back to VW investor-relations page…
10:39:52  INFO      VW IR: scraping investor-relations page for 2025…



✓  8 filing(s) for 'Volkswagen' (2024) → financial_reports\Volkswagen\2024


10:39:56  INFO      VW IR: 8 report(s) found for 2025
10:40:02  INFO          saved: VW_Konzernabschluss_2025.pdf
10:40:06  INFO          saved: VW_Jahresabschluss_2025.pdf
10:40:10  INFO          saved: VW_Konzernabschluss_2025.htm
10:40:13  INFO          saved: VW_Jahresabschluss_2025.htm
10:40:17  INFO          skip (exists): VW_Konzernabschluss_2025.pdf
10:40:21  INFO          skip (exists): VW_Jahresabschluss_2025.pdf
10:40:25  INFO          skip (exists): VW_Konzernabschluss_2025.htm
10:40:29  INFO          skip (exists): VW_Jahresabschluss_2025.htm



✓  8 filing(s) for 'Volkswagen' (2025) → financial_reports\Volkswagen\2025

Done — 32 file(s) saved in total.
